# Hard peak-cancellation CFR on the XCZU67DR

One digital 20 MHz carrier at 100 MS/s. The 16-bit source uses independent random subcarrier phases for each of 64 periodic symbols. Each has 4,096 samples. Repeating the symbol four times permits steady-state capture and integer-lag alignment. The calibrated low-amplitude constant-envelope sequence supplies the lag for every measurement. A fixed 249-tap Q14 pulse on a fourfold fractional-sample grid follows the carrier bandwidth.

In [ ]:
import json, os, sys
from pathlib import Path
from IPython.display import display, Markdown, Image
example = Path(os.getenv('CFR_EXAMPLE', '../packages/pynq-dfe/examples/cfr')).resolve()
sys.path.insert(0, str(example))
from experiment import acquire, plot
output = Path(os.getenv('CFR_OUTPUT', 'cfr-results')).resolve()
output.mkdir(exist_ok=True)
frozen = (example / 'expectations.json').read_text()
(output / 'expectations.json').write_text(frozen)
print(frozen)

The PC and window thresholds are 12,000 counts, −8.73 dBFS or 6.021 dB above input mean power. The model subtracts a bandwidth-matched pulse at each local maximum in three passes. It approximates stage scheduling and the window postprocessor. Each CCDF point has a ±0.8 dB model tolerance. The 0.01% tail must also stay below 6.521 dB, each adjacent channel must be at least 35 dB below the occupied channel, and the low-level error must stay within three counts in complex magnitude. The ACLR limit is an experiment mask. EVM uses the in-band power of the aligned sample error without fitting gain or phase.

In [ ]:
result = acquire(Path(os.environ['CFR_BITFILE']), output)
print('Lag (samples modulo 4096)', result['latency_mod_period'])
print('Before and after', json.dumps([result['before'], result['after']], indent=2))
table = ['| Test | Expected | Measured | Result |', '|---|---|---:|---|']
for row in result['rows']:
    status = 'PASS' if row['passed'] else 'FAIL'
    table.append(f"| {row['test']} | {row['expected']} | {row['measured']:.4f} | {status} |")
display(Markdown('\n'.join(table)))

In [ ]:
import matplotlib.pyplot as plt
fig = plot(output)
plt.close(fig)
display(Image(filename=str(output/'ccdf-spectrum.png')))

The left plot uses mean-normalized instantaneous power in dB on a linear horizontal axis and a logarithmic probability axis. The right plot shows the complex baseband spectrum at linear frequency spacing. ACLR integrates the two 20 MHz channels immediately beside the occupied band.